# Part 4 — Bias Mitigation

Self-contained notebook. Re-runs the data load and (where needed) reloads the saved baseline checkpoint so it can execute standalone on Colab.

In [ ]:
!pip install -q transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

USE_COLS = ["comment_text", "toxic", "black", "white", "muslim", "jewish"]
IDENTITY_COLS = ["black", "white", "muslim", "jewish"]

raw = pd.read_csv("data/jigsaw-unintended-bias-train.csv", usecols=USE_COLS)
raw["label"] = (raw["toxic"] >= 0.5).astype(int)

sample = raw.sample(n=120_000, random_state=SEED)
train_df, eval_df = train_test_split(
    sample, test_size=20_000, stratify=sample["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
print("train:", train_df.shape, "eval:", eval_df.shape)


In [ ]:
import os, glob
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

# Prefer best mitigated checkpoint if present (Part 5 scenario), else baseline,
# else fall back to the pretrained backbone.
_candidates = (sorted(glob.glob("checkpoints/best_mitigated_*"))
               + ["checkpoints/baseline"])
CKPT_DIR = next((c for c in _candidates if os.path.isdir(c)), MODEL_NAME)
print("loading model from", CKPT_DIR)

tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR, num_labels=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

class ToxicDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(df["comment_text"].astype(str).tolist(),
                        truncation=True, padding="max_length", max_length=MAX_LEN)
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = df["label"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.input_ids[i]),
            "attention_mask": torch.tensor(self.attention_mask[i]),
            "labels": torch.tensor(self.labels[i]),
        }

train_ds = ToxicDataset(train_df)
eval_ds = ToxicDataset(eval_df)


In [ ]:
# Score the eval set so downstream cells can reuse predictions.
from transformers import Trainer, TrainingArguments

_args = TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=64,
                          report_to="none")
_trainer = Trainer(model=model, args=_args)
_logits = _trainer.predict(eval_ds).predictions
eval_probs = torch.softmax(torch.tensor(_logits), dim=-1)[:, 1].numpy()
eval_preds = (eval_probs >= 0.5).astype(int)
eval_df = eval_df.assign(prob=eval_probs, pred=eval_preds)
trainer = _trainer  # downstream cells reference `trainer` and `metrics`
metrics = _trainer.evaluate(eval_ds)
print({k: round(v, 4) for k, v in metrics.items() if k.startswith("eval_")})


In [ ]:
# Shared helpers reused across multiple parts (defined once here so each
# notebook is fully standalone).
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score)

def compute_metrics(pred):
    logits, labels = pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "auc_roc": roc_auc_score(labels, probs),
    }

def per_cohort_metrics(df):
    y, p = df["label"].values, df["pred"].values
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "n": len(df),
        "TPR": tp / (tp + fn) if (tp + fn) else 0.0,
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "FNR": fn / (tp + fn) if (tp + fn) else 0.0,
        "Precision": precision_score(y, p, zero_division=0),
    }


In [ ]:
def audit(eval_with_preds: pd.DataFrame) -> dict:
    """Compute the bias-audit summary row used in the comparison table.
    Expects an `eval_df`-shaped frame with `label`, `pred`, `prob`, identity cols.
    """
    hb = eval_with_preds[eval_with_preds["black"] >= 0.5]
    rf = eval_with_preds[(eval_with_preds["white"] >= 0.5) & (eval_with_preds["black"] < 0.1)]
    hb_m, rf_m = per_cohort_metrics(hb), per_cohort_metrics(rf)

    audit_df = pd.concat([hb.assign(group=1), rf.assign(group=0)])[["label", "pred", "group"]]
    bld_t = BinaryLabelDataset(df=audit_df[["label", "group"]].rename(columns={"label": "y"}),
                               label_names=["y"], protected_attribute_names=["group"])
    bld_p = bld_t.copy(); bld_p.labels = audit_df["pred"].values.reshape(-1, 1)
    cm = ClassificationMetric(bld_t, bld_p,
                              unprivileged_groups=[{"group": 1}],
                              privileged_groups=[{"group": 0}])

    overall_f1 = f1_score(eval_with_preds["label"], eval_with_preds["pred"], average="macro")
    return {
        "F1_macro": round(overall_f1, 4),
        "FPR_high_black": round(hb_m["FPR"], 4),
        "FPR_reference": round(rf_m["FPR"], 4),
        "StatParityDiff": round(cm.statistical_parity_difference(), 4),
        "EqOppDiff": round(cm.equal_opportunity_difference(), 4),
    }

baseline_row = audit(eval_df)
print("baseline:", baseline_row)

In [ ]:
# --- 1. Reweighing (AIF360) -------------------------------------------------
from aif360.algorithms.preprocessing import Reweighing

# Build a protected-attribute column on train: 1 = high-black, 0 = reference, NaN = neither.
train_df["group"] = np.where(train_df["black"] >= 0.5, 1,
                      np.where((train_df["white"] >= 0.5) & (train_df["black"] < 0.1), 0, np.nan))
rw_subset = train_df.dropna(subset=["group"]).copy()
rw_subset["group"] = rw_subset["group"].astype(int)

bld = BinaryLabelDataset(
    df=rw_subset[["label", "group"]].rename(columns={"label": "y"}),
    label_names=["y"], protected_attribute_names=["group"],
)
rw = Reweighing(unprivileged_groups=[{"group": 1}], privileged_groups=[{"group": 0}])
bld_rw = rw.fit_transform(bld)

# Map the AIF360 weights back; rows outside both cohorts get weight 1.0.
weight_lookup = pd.Series(bld_rw.instance_weights, index=rw_subset.index)
train_df["sample_weight"] = weight_lookup.reindex(train_df.index).fillna(1.0).values
print(train_df.groupby(train_df["group"].fillna(-1))["sample_weight"].mean().round(3))

In [ ]:
# Retrain DistilBERT with per-sample weights via a custom Trainer subclass.
class WeightedDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(df["comment_text"].astype(str).tolist(),
                        truncation=True, padding="max_length", max_length=MAX_LEN)
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = df["label"].tolist()
        self.weights = df["sample_weight"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.input_ids[i]),
            "attention_mask": torch.tensor(self.attention_mask[i]),
            "labels": torch.tensor(self.labels[i]),
            "sample_weight": torch.tensor(self.weights[i], dtype=torch.float),
        }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # eval batches won't carry sample_weight; fall back to uniform weights.
        weights = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, reduction="none")
        if weights is not None:
            loss = loss * weights
        loss = loss.mean()
        return (loss, outputs) if return_outputs else loss

rw_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
rw_args = TrainingArguments(
    output_dir="checkpoints/reweighed",
    num_train_epochs=3, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="no", save_strategy="no",
    logging_steps=200, seed=SEED, report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,  # keep `sample_weight` in the batch
)
rw_trainer = WeightedTrainer(
    model=rw_model, args=rw_args,
    train_dataset=WeightedDataset(train_df), eval_dataset=eval_ds,
)
rw_trainer.train()

rw_logits = rw_trainer.predict(eval_ds).predictions
rw_probs = torch.softmax(torch.tensor(rw_logits), dim=-1)[:, 1].numpy()
eval_rw = eval_df.assign(prob=rw_probs, pred=(rw_probs >= 0.5).astype(int))
reweigh_row = audit(eval_rw)
print("reweighing:", reweigh_row)

In [ ]:
# --- 2. ThresholdOptimizer (Fairlearn) -------------------------------------
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.base import BaseEstimator, ClassifierMixin

# Simple sklearn-style wrapper: feature X is the model's probability itself,
# so predict_proba is just [1-X, X]. Avoids any index plumbing.
class IdentityProbModel(BaseEstimator, ClassifierMixin):
    classes_ = np.array([0, 1])
    def fit(self, X, y): return self
    def predict(self, X): return (X.ravel() >= 0.5).astype(int)
    def predict_proba(self, X):
        p = X.ravel()
        return np.vstack([1 - p, p]).T

# Restrict to the two cohorts so the protected attribute is well-defined.
mask = (eval_df["black"] >= 0.5) | ((eval_df["white"] >= 0.5) & (eval_df["black"] < 0.1))
sub = eval_df.loc[mask].reset_index(drop=True)
X_to = sub["prob"].values.reshape(-1, 1)
y_to = sub["label"].values
sensitive = np.where(sub["black"] >= 0.5, "high_black", "reference")

to = ThresholdOptimizer(
    estimator=IdentityProbModel(), constraints="equalized_odds",
    objective="accuracy_score", prefit=True, predict_method="predict_proba",
)
to.fit(X_to, y_to, sensitive_features=sensitive)
to_preds = to.predict(X_to, sensitive_features=sensitive, random_state=SEED)

eval_to = sub.assign(pred=to_preds)
threshopt_row = audit(eval_to)
print("threshold_opt:", threshopt_row)

In [ ]:
# --- 3. Oversampling (data-level) ------------------------------------------
hb_train = train_df[train_df["black"] >= 0.5]
oversampled_df = pd.concat([train_df] + [hb_train] * 3, ignore_index=True)
print(f"train rows: {len(train_df)} -> {len(oversampled_df)} (added {3*len(hb_train)})")

os_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
os_args = TrainingArguments(
    output_dir="checkpoints/oversampled",
    num_train_epochs=3, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="no", save_strategy="no",
    logging_steps=200, seed=SEED, report_to="none",
    fp16=torch.cuda.is_available(),
)
os_trainer = Trainer(
    model=os_model, args=os_args,
    train_dataset=ToxicDataset(oversampled_df), eval_dataset=eval_ds,
)
os_trainer.train()

os_logits = os_trainer.predict(eval_ds).predictions
os_probs = torch.softmax(torch.tensor(os_logits), dim=-1)[:, 1].numpy()
eval_os = eval_df.assign(prob=os_probs, pred=(os_probs >= 0.5).astype(int))
oversample_row = audit(eval_os)
print("oversampling:", oversample_row)

In [ ]:
# --- Accuracy-fairness Pareto: sweep per-cohort thresholds -----------------
# Fairlearn's ThresholdOptimizer doesn't expose a tolerance parameter, so we
# sweep a 2D grid of (threshold_high_black, threshold_reference), compute
# overall F1 + equal opportunity difference for each pair, and plot the cloud
# with the Pareto frontier overlaid.
hb_idx = eval_df.index[eval_df["black"] >= 0.5]
rf_idx = eval_df.index[(eval_df["white"] >= 0.5) & (eval_df["black"] < 0.1)]
y_all = eval_df["label"].values

grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
points = []
for t_hb in grid:
    for t_rf in grid:
        preds = (eval_df["prob"].values >= 0.5).astype(int)  # default elsewhere
        preds[hb_idx] = (eval_df.loc[hb_idx, "prob"].values >= t_hb).astype(int)
        preds[rf_idx] = (eval_df.loc[rf_idx, "prob"].values >= t_rf).astype(int)
        # EqOppDiff = TPR(unpriv) - TPR(priv) on the two cohorts.
        def tpr(idx):
            y, p = y_all[idx], preds[idx]
            pos = (y == 1)
            return (p[pos] == 1).mean() if pos.any() else 0.0
        eod = tpr(hb_idx) - tpr(rf_idx)
        f1 = f1_score(y_all, preds, average="macro")
        points.append({"t_hb": t_hb, "t_rf": t_rf, "EqOppDiff": eod, "F1_macro": f1})
pareto_df = pd.DataFrame(points)
print("grid points:", len(pareto_df))

In [ ]:
# Pareto frontier: keep points that aren't dominated by any other point on
# (lower |EqOppDiff|, higher F1). Plot the full cloud + frontier in red.
pareto_df["abs_eod"] = pareto_df["EqOppDiff"].abs()
sorted_df = pareto_df.sort_values("abs_eod").reset_index(drop=True)
frontier, best_f1 = [], -1.0
for _, row in sorted_df.iterrows():
    if row["F1_macro"] > best_f1:
        frontier.append(row); best_f1 = row["F1_macro"]
frontier_df = pd.DataFrame(frontier)

plt.figure(figsize=(7, 5))
plt.scatter(pareto_df["abs_eod"], pareto_df["F1_macro"], s=10, alpha=0.3, label="grid")
plt.plot(frontier_df["abs_eod"], frontier_df["F1_macro"], "r-o", ms=5, label="Pareto frontier")
plt.xlabel("|Equal Opportunity Difference|  (lower = fairer)")
plt.ylabel("Overall F1 (macro)")
plt.title("Accuracy-fairness Pareto: per-cohort threshold sweep")
plt.legend(); plt.tight_layout(); plt.show()

print("\nfrontier sample (5 points):")
print(frontier_df[["t_hb", "t_rf", "EqOppDiff", "F1_macro"]].round(4).head().to_string(index=False))

In [ ]:
comparison = pd.DataFrame({
    "baseline": baseline_row,
    "reweighing": reweigh_row,
    "threshold_opt": threshopt_row,
    "oversampling": oversample_row,
}).T
print(comparison)

# Quick note on the trade-off: anything that pushes EqOppDiff toward 0 usually
# costs F1. Read the table left-to-right to see which strategy gives the best
# fairness/accuracy balance on this run.

In [ ]:
# Demographic parity vs equalized odds — base rates
hb_base = (eval_df.loc[eval_df["black"] >= 0.5, "label"] == 1).mean()
rf_base = (eval_df.loc[(eval_df["white"] >= 0.5) & (eval_df["black"] < 0.1), "label"] == 1).mean()
print(f"base rate (toxic prevalence)  high_black={hb_base:.4f}  reference={rf_base:.4f}")
print(f"absolute base-rate gap = {abs(hb_base - rf_base):.4f}")

### Can you get demographic parity AND equalized odds at the same time?

**No, not generally — and the cell above shows why for this dataset.**

- **Demographic parity** requires `P(ŷ=1 | A=high_black) == P(ŷ=1 | A=reference)` — the same flag rate in both groups.
- **Equalized odds** requires `P(ŷ=1 | y=1, A=·)` and `P(ŷ=1 | y=0, A=·)` to be equal across groups — the same TPR and FPR.

When the two groups have different **base rates** `p_a = P(y=1|A=a)`, any classifier with equal TPR `t` and equal FPR `f` produces a positive prediction rate of `p_a · t + (1 - p_a) · f`, which depends on `p_a`. So equal TPR + FPR ⇒ unequal prediction rates whenever base rates differ. The two definitions are mutually satisfiable only when (a) base rates are equal, or (b) the classifier is perfect.

The printed gap above is non-zero, so on this dataset you must pick which fairness definition to satisfy.

In [ ]:
# Save best mitigated model: choose the row with the smallest |EqOppDiff|
# while keeping F1 within 2 pts of baseline. Falls back to lowest |EqOppDiff|.
candidates = {"reweighing": (rw_trainer.model, reweigh_row),
              "oversampling": (os_trainer.model, oversample_row)}
ranked = sorted(candidates.items(), key=lambda kv: abs(kv[1][1]["EqOppDiff"]))
best_name, (best_model, best_row) = ranked[0]
print(f"best mitigated model: {best_name}  ({best_row})")

best_dir = f"checkpoints/best_mitigated_{best_name}"
best_model.save_pretrained(best_dir)
tokenizer.save_pretrained(best_dir)
print("saved to", best_dir)